# IMDB Sentiment Analysis

# Import Libraries

In [143]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from keras.datasets import imdb
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import Flatten
from keras.layers import Conv1D
from keras.layers import MaxPooling1D
from keras.layers import Embedding

import nltk
from nltk.corpus import stopwords

# Clean the Data

Load the dataset

Keep the top n words and zero the rest (n = 5000)

Split data into training and testing sets

In [144]:
# Load the IMDB dataset from keras, limiting to the top 5000 most frequent words
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=5000)

# Sanity Check: Print the first several lines of the dataset
print("First 3 training samples (as word indices):")
for i in range(3):
    print(X_train[i])
    print("Label:", y_train[i])

First 3 training samples (as word indices):
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 2, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 2, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 2, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 2, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 2, 19, 178, 32]
Label: 1
[1, 194, 1153, 194, 2,

Pad dataset to a maximum review length in words

In [145]:
# Find max length of reviews in test set
# max_length = max(len(review) for review in X_test)
# print(f"Max length of reviews in test set: {max_length}")

# Max length was found in the training set so this is just commented out

In [146]:
# Pad each review to match the largest review length

# Find the maximum length of the reviews in the training set
max_length = max(len(review) for review in X_train)

# Pad the sequences to ensure uniform length
X_train = keras.preprocessing.sequence.pad_sequences(X_train, maxlen=max_length, padding='post')
X_test = keras.preprocessing.sequence.pad_sequences(X_test, maxlen=max_length, padding='post')

In [147]:
# Sanity Check: Print the shape of the padded data
print("Shape of training data:", X_train.shape)
print("Shape of testing data:", X_test.shape)

Shape of training data: (25000, 2494)
Shape of testing data: (25000, 2494)


# Predict Positive/Negative

Use logistic regression model and feed-forward neural network model.

## Logistic Regression Model

In [148]:
def lr_model():
    model = Sequential([
        Embedding(input_dim=5000, output_dim=32),
        Flatten(),
        Dense(2, activation='softmax')
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [149]:
# Create the logistic regression model
logistic_model = lr_model()

# Train the model
logistic_model.fit(X_train, y_train, epochs=5, batch_size=128, validation_split=0.2)

# Evaluate the model
logistic_loss, logistic_accuracy = logistic_model.evaluate(X_test, y_test)
print(f"\n\nLogistic Regression Model - Loss: {logistic_loss}, Accuracy: {logistic_accuracy}")

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 45ms/step - accuracy: 0.5520 - loss: 0.8146 - val_accuracy: 0.8110 - val_loss: 0.4304
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step - accuracy: 0.8551 - loss: 0.3490 - val_accuracy: 0.8578 - val_loss: 0.3353
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 6s 41ms/step - accuracy: 0.9150 - loss: 0.2317 - val_accuracy: 0.8500 - val_loss: 0.3485
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 6s 41ms/step - accuracy: 0.9468 - loss: 0.1664 - val_accuracy: 0.8716 - val_loss: 0.3230
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.9735 - loss: 0.1097 - val_accuracy: 0.8676 - val_loss: 0.3364
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8588 - loss: 0.3384


Logistic Regression Model - Loss: 0.34186336398124695, Accuracy: 0.8576800227165222


## Feed-Forward Neural Network Model

In [150]:
def ff_model():
    model = Sequential([
        Embedding(input_dim=5000, output_dim=32),  # Embedding layer
        Flatten(),  # Flatten the output of the embedding layer
        Dense(1000, activation='relu'),
        Dense(200, activation='relu'),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [151]:
# Create the feed-forward neural network model
ffnn_model = ff_model()

# Train the model
ffnn_model.fit(X_train, y_train, epochs=5, batch_size=128, validation_split=0.2)

# Evaluate the model
ffnn_loss, ffnn_accuracy = ffnn_model.evaluate(X_test, y_test)
print(f"\n\nFeed-Forward Neural Network Model - Loss: {ffnn_loss}, Accuracy: {ffnn_accuracy}")

Epoch 1/5
 13/157 ━━━━━━━━━━━━━━━━━━━━ 1:49 762ms/step - accuracy: 0.5137 - loss: 2.9326

KeyboardInterrupt: 

# Convolutional Neural Network

- Use at least one Conv1D and MaxPooling1D in the model.
- Make best judgement for activation function and optimizer
- Use accuracy for metrics

In [ ]:
def cnn_model():
    model = Sequential([
        Embedding(input_dim=5000, output_dim=32),
        Conv1D(filters=32, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.2),
        Flatten(),
        Dense(1000, activation='relu'),
        Dense(200, activation='relu'),
        Dense(32, activation='relu'),
        Dense(1, activation='softmax')
    ])

    # Summary
    model.summary()

    # Compile the model
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# Create the CNN model
convolutional_model = cnn_model()

Model: "sequential_42"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_37 (Embedding)        │ (None, 2494, 32)       │       160,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ (None, 2492, 32)       │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_10 (MaxPooling1D) │ (None, 1246, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 1246, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_33 (Flatten)            │ (None, 39872)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_86 (Dense)                │ (None, 1000)           │    39,873,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_87 (Dense)                │ (None, 200)            │       200,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_88 (Dense)                │ (None, 32)             │         6,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_89 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,242,769 (153.51 MB)

 Trainable params: 40,242,769 (153.51 MB)

 Non-trainable params: 0 (0.00 B)

## Fit CNN

In [ ]:
# Train the model
convolutional_model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 137s 433ms/step - accuracy: 0.6026 - loss: 0.7206 - val_accuracy: 0.8626 - val_loss: 0.3216
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 133s 424ms/step - accuracy: 0.9123 - loss: 0.2308 - val_accuracy: 0.8698 - val_loss: 0.3132
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 130s 416ms/step - accuracy: 0.9465 - loss: 0.1562 - val_accuracy: 0.8778 - val_loss: 0.3239
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 131s 419ms/step - accuracy: 0.9661 - loss: 0.0987 - val_accuracy: 0.8758 - val_loss: 0.3744
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 129s 412ms/step - accuracy: 0.9864 - loss: 0.0432 - val_accuracy: 0.8774 - val_loss: 0.4529


## Evaluate

Print accuracy in percentage (make best judgement for epochs, batch size)

In [ ]:
# Evaluate the model
cnn_loss, cnn_accuracy = convolutional_model.evaluate(X_test, y_test)
print(f"\n\nConvolutional Neural Network Model - Loss: {cnn_loss}, Accuracy: {cnn_accuracy}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8621 - loss: 0.4305


Convolutional Neural Network Model - Loss: 0.430576354265213, Accuracy: 0.8619599938392639


# Remove Stopwords

Rerun and check performance.    

In [ ]:
# Load the stopwords from NLTK
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Sanity Check: Print the first few stopwords
print("First 10 stopwords:", list(stop_words)[:10])

First 10 stopwords: ['on', 'my', 'having', 'same', "aren't", 'hers', 'where', 'll', "they'd", 'that']


[nltk_data] Downloading package stopwords to /home/kevin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Get mapping of word indices to words
word_index = imdb.get_word_index()

# Explore the word index
print("First 10 words in the word index:")
for i, (word, index) in enumerate(word_index.items()):
    if i < 10:
        print(f"{word}: {index}")
    else:
        break

First 10 words in the word index:
fawn: 34701
tsukino: 52006
nunnery: 52007
sonja: 16816
vani: 63951
woods: 1408
spiders: 16115
hanging: 2345
woody: 2289
trawling: 52008


In [ ]:
# Create a python Set of indices for stopwords in the word index
stopword_indices = {index for word, index in word_index.items() if word in stop_words}

# Add padding 0's to the stopword indices
stopword_indices.add(0)

In [ ]:
# Remove stopwords from training and testing reviews
def remove_stopwords(reviews, stopword_indices):
    filtered = []
    for review in reviews:
        filtered_review = [word for word in review if word not in stopword_indices]
        filtered.append(filtered_review)
    return filtered

# Filter the training and testing reviews
X_train_filtered = remove_stopwords(X_train, stopword_indices)
X_test_filtered = remove_stopwords(X_test, stopword_indices)

# Find the maximum length of the filtered reviews in training and testing sets
max_length_filtered = max(max(len(review) for review in X_train_filtered), max(len(review) for review in X_test_filtered))

# Pad the filtered sequences to ensure uniform length
X_train_filtered = keras.preprocessing.sequence.pad_sequences(X_train_filtered, maxlen=max_length_filtered, padding='post')
X_test_filtered = keras.preprocessing.sequence.pad_sequences(X_test_filtered, maxlen=max_length_filtered, padding='post')

# Sanity Check: Print the shape of the filtered padded data
print("Shape of filtered training data:", X_train_filtered.shape)
print("Shape of filtered testing data:", X_test_filtered.shape)

Shape of filtered training data: (25000, 1068)
Shape of filtered testing data: (25000, 1068)


In [ ]:
# Retrain and Evaluate Logistic Regression Model with Stopwords Removed
logistic_model_filtered = lr_model()
logistic_model_filtered.fit(X_train_filtered, y_train, epochs=5, batch_size=128, validation_split=0.2)

# Evaluate the model
logistic_loss_filtered, logistic_accuracy_filtered = logistic_model_filtered.evaluate(X_test_filtered, y_test)
print(f"\n\nLogistic Regression Model (Stopwords Removed) - Loss: {logistic_loss_filtered}, Accuracy: {logistic_accuracy_filtered}")

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.5814 - loss: 0.6665 - val_accuracy: 0.8266 - val_loss: 0.3922
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.8694 - loss: 0.3279 - val_accuracy: 0.8716 - val_loss: 0.3107
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9145 - loss: 0.2268 - val_accuracy: 0.8714 - val_loss: 0.3118
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9332 - loss: 0.1860 - val_accuracy: 0.8718 - val_loss: 0.3172
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.9538 - loss: 0.1458 - val_accuracy: 0.8722 - val_loss: 0.3250
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8582 - loss: 0.3424


Logistic Regression Model (Stopwords Removed) - Loss: 0.339885950088501, Accuracy: 0.8585600256919861


In [ ]:
# Retrain and Evaluate the feed-forward neural network model with stopwords removed
ffnn_model_filtered = ff_model()
ffnn_model_filtered.fit(X_train_filtered, y_train, epochs=5, batch_size=64, validation_split=0.2)

# Evaluate the model
ffnn_loss_filtered, ffnn_accuracy_filtered = ffnn_model_filtered.evaluate(X_test_filtered, y_test)
print(f"\n\nFeed-Forward Neural Network Model (Stopwords Removed) - Loss: {ffnn_loss_filtered}, Accuracy: {ffnn_accuracy_filtered}")

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 95s 294ms/step - accuracy: 0.5751 - loss: 0.9129 - val_accuracy: 0.8548 - val_loss: 0.3423
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 91s 290ms/step - accuracy: 0.8295 - loss: 0.6988 - val_accuracy: 0.8564 - val_loss: 0.3371
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 91s 290ms/step - accuracy: 0.9037 - loss: 0.2688 - val_accuracy: 0.8306 - val_loss: 0.3800
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 92s 294ms/step - accuracy: 0.9290 - loss: 0.2093 - val_accuracy: 0.8552 - val_loss: 0.3650
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 93s 296ms/step - accuracy: 0.9766 - loss: 0.0831 - val_accuracy: 0.8528 - val_loss: 0.4780
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.8327 - loss: 0.5363


Feed-Forward Neural Network Model (Stopwords Removed) - Loss: 0.5370347499847412, Accuracy: 0.8314800262451172


In [ ]:
# Retrain and Evaluate the CNN model with stopwords removed
convolutional_model_filtered = cnn_model()
convolutional_model_filtered.fit(X_train_filtered, y_train, epochs=5, batch_size=64, validation_split=0.2)

# Evaluate the model
cnn_loss_filtered, cnn_accuracy_filtered = convolutional_model_filtered.evaluate(X_test_filtered, y_test)
print(f"\n\nConvolutional Neural Network Model (Stopwords Removed) - Loss: {cnn_loss_filtered}, Accuracy: {cnn_accuracy_filtered}")

Model: "sequential_46"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_41 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_11 (MaxPooling1D) │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_37 (Flatten)            │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_96 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_97 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_98 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_99 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 61s 189ms/step - accuracy: 0.6312 - loss: 0.5876 - val_accuracy: 0.8596 - val_loss: 0.3313
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 55s 175ms/step - accuracy: 0.9111 - loss: 0.2360 - val_accuracy: 0.8656 - val_loss: 0.3132
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 57s 183ms/step - accuracy: 0.9440 - loss: 0.1548 - val_accuracy: 0.8674 - val_loss: 0.3497
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 55s 176ms/step - accuracy: 0.9755 - loss: 0.0751 - val_accuracy: 0.8612 - val_loss: 0.4385
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 57s 183ms/step - accuracy: 0.9866 - loss: 0.0392 - val_accuracy: 0.8552 - val_loss: 0.5815
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8494 - loss: 0.6169


Convolutional Neural Network Model (Stopwords Removed) - Loss: 0.6030154228210449, Accuracy: 0.8511199951171875
